# Carregando os dados

## Inicialzação

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

In [3]:

df_estados = pd.read_csv(
    '/content/estados_id.csv',
)

In [4]:
df_municipios = pd.read_csv(
    '/content/municipios_id.csv',
)

In [5]:
df_municipios_com_estado = pd.merge(df_estados[['estado_id',	'uf']], df_municipios, on='estado_id', how='inner')

In [6]:
df_emendas_favorecido = pd.read_csv('/content/drive/MyDrive/9 período/descritiva/EmendasParlamentares_PorFavorecido.csv',  encoding='latin1',
    sep=';')

In [7]:
df_emendas_favorecido = df_emendas_favorecido[df_emendas_favorecido['Código da Emenda'] != 'Sem informação']

In [8]:
len(df_emendas_favorecido)

716604

In [9]:
df_emendas_favorecido = df_emendas_favorecido[df_emendas_favorecido['Município Favorecido'].isna() == False]

In [10]:
len(df_emendas_favorecido)

715322

In [11]:
df_emendas_favorecido["Valor Recebido"] = (
    df_emendas_favorecido["Valor Recebido"]
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

In [12]:
df_emendas = (
    df_emendas_favorecido.groupby(
        ["Código da Emenda", "Município Favorecido", "UF Favorecido"],
        as_index=False
    )
    .agg({
        "Valor Recebido": "sum",
        "Ano/Mês": "first"
    })
)

In [13]:
import unicodedata
import re

def transformar_nome_cidade(texto):

    texto_sem_acentos = unicodedata.normalize(
        'NFKD',
        str(texto)
    ).encode(
        'ASCII',
        'ignore'
    ).decode('ASCII')

    # remove apostrofos
    texto_sem_acentos = texto_sem_acentos.replace("'", "")

    # troca espaços por hífen
    resultado = texto_sem_acentos.replace(" ", "-")

    # remove caracteres especiais restantes
    resultado = re.sub(r'[^a-zA-Z0-9\-]', '', resultado)

    return resultado.lower().strip()

In [14]:
df_municipios_com_estado['nome'] = df_municipios_com_estado['nome'].apply(transformar_nome_cidade)
df_emendas['Município Favorecido'] = df_emendas['Município Favorecido'].apply(transformar_nome_cidade)

In [15]:
len(df_emendas)

300953

In [16]:
df_emendas = df_emendas[df_emendas['UF Favorecido'].isna() == False]
df_emendas = df_emendas[df_emendas['Município Favorecido'].isna() == False]
len(df_emendas)

300953

In [17]:
df_emendas_com_municipio = pd.merge(df_municipios_com_estado, df_emendas, left_on=['nome', 'uf'], right_on=['Município Favorecido', 'UF Favorecido'], how='inner')
len(df_emendas_com_municipio)

300063

In [18]:
len(df_emendas_com_municipio['municipio_id'].unique())

5550

In [19]:
df_panorama = pd.read_csv('/content/drive/MyDrive/9 período/descritiva/dados_municipios_panorama.csv')

In [20]:
df_panorama['municipio'] = df_panorama['municipio'].apply(transformar_nome_cidade)

In [21]:
df_dados = pd.merge(df_emendas_com_municipio, df_panorama, on='municipio_id', how='inner')
len(df_dados)

299961

In [22]:
colunas_df_dados = [
    "estado_id",
    #"uf",
    'Ano/Mês',
    "municipio_id",
    #"nome",
    "Código da Emenda",
    #"Município Favorecido",
    #"UF Favorecido",
    "Valor Recebido",
    #"municipio",
    "População no último censo",
    #"População estimada",
    "População quilombola",
    "População indígena",
    "Densidade demográfica",
    "Nome masculino mais popular",
    "Nome feminino mais popular",
    "Sobrenome mais popular",
    "Salário médio mensal dos trabalhadores formais",
    "Pessoal ocupado em postos de trabalho formais",
    "Percentual da população com rendimento nominal mensal per capita de até 1/2 salário mínimo",
    "Taxa de escolarização de 6 a 14 anos de idade",
    "IDEB – Anos iniciais do ensino fundamental (Rede pública)",
    "IDEB – Anos finais do ensino fundamental (Rede pública)",
    "Matrículas no ensino fundamental",
    "Matrículas no ensino médio",
    "Docentes no ensino fundamental",
    "Docentes no ensino médio",
    "Número de estabelecimentos de ensino fundamental",
    "Número de estabelecimentos de ensino médio",
    "PIB per capita",
    "Índice de Desenvolvimento Humano Municipal (IDHM)",
    "Total de receitas brutas realizadas",
    "Transferências correntes (Percentual em relação às receitas correntes brutas realizadas)",
    "Total de despesas brutas empenhadas",
    "Mortalidade Infantil",
    "Internações por diarreia pelo SUS",
    "Estabelecimentos de Saúde SUS",
    "Área urbanizada",
    "Esgotamento sanitário por rede geral, rede pluvial ou fossa ligada à rede",
    "Arborização de vias públicas",
    "Urbanização de vias públicas",
   # "População exposta ao risco",
    "Bioma predominante",
    "Sistema Costeiro-Marinho",
    "Área da unidade territorial",
    "Hierarquia urbana",
    "Região de Influência",
    "Região intermediária",
    "Região imediata",
    "Mesorregião",
    "Microrregião"
]

In [23]:
df_dados = df_dados[colunas_df_dados]

## Tratando os dados

In [24]:
def turn_value_to_international_stardand(value):
  if isinstance(value, int):
    return value
  if isinstance(value, float):
    return value

  valor = value.replace(".", "")
  valor = valor.replace(",", ".")

  return float(valor)

def categorizar_valores(valores):
  try:
    return pd.qcut(
      valores,
      q=3,
      labels=["valor_baixo", "valor_medio", "valor_alto"],
  ).tolist()
  except:
     return pd.cut(
      valores,
       bins=3,
      labels=["valor_baixo", "valor_medio", "valor_alto"],
  ).tolist()


In [25]:
df_dados.loc[:, 'Valor Recebido'] = (
    df_dados['Valor Recebido']
    .apply(turn_value_to_international_stardand)
)
df_dados = df_dados[df_dados['Valor Recebido'] > 0]

In [26]:
df_dados['Valor Recebido'] = categorizar_valores(list(df_dados['Valor Recebido']))

In [27]:
def remove_barra(value):
  return 0 if value == '-' else value

In [28]:
colunas_numericas = [
    "População no último censo",
    "População quilombola",
    "População indígena",
    "Densidade demográfica",
    "Salário médio mensal dos trabalhadores formais",
    "Pessoal ocupado em postos de trabalho formais",
    "Percentual da população com rendimento nominal mensal per capita de até 1/2 salário mínimo",
    "Taxa de escolarização de 6 a 14 anos de idade",
    "IDEB – Anos iniciais do ensino fundamental (Rede pública)",
    "IDEB – Anos finais do ensino fundamental (Rede pública)",
    "Matrículas no ensino fundamental",
    "Matrículas no ensino médio",
    "Docentes no ensino fundamental",
    "Docentes no ensino médio",
    "Número de estabelecimentos de ensino fundamental",
    "Número de estabelecimentos de ensino médio",
    "PIB per capita",
    "Índice de Desenvolvimento Humano Municipal (IDHM)",
    "Total de receitas brutas realizadas",
    "Transferências correntes (Percentual em relação às receitas correntes brutas realizadas)",
    "Total de despesas brutas empenhadas",
    "Mortalidade Infantil",
    "Internações por diarreia pelo SUS",
    "Estabelecimentos de Saúde SUS",
    "Área urbanizada",
    "Esgotamento sanitário por rede geral, rede pluvial ou fossa ligada à rede",
    "Arborização de vias públicas",
    "Urbanização de vias públicas",
    "Área da unidade territorial"
]

In [29]:
for coluna in colunas_numericas:
  if len(df_dados[df_dados[coluna] == 'Sem dados']) > 0:
    df_dados = df_dados.drop(columns=[coluna])
    continue

  df_dados.loc[:, coluna] = (
      df_dados[coluna]
      .apply(remove_barra)
  )
  df_dados.loc[:, coluna] = (
      df_dados[coluna]
      .apply(turn_value_to_international_stardand)
  )

In [30]:
colunas_per_capita = [
    "População quilombola",
    "População indígena",
    "Pessoal ocupado em postos de trabalho formais",
    "Matrículas no ensino fundamental",
    "Matrículas no ensino médio",
    "Docentes no ensino fundamental",
    "Docentes no ensino médio",
    "Número de estabelecimentos de ensino fundamental",
    "Número de estabelecimentos de ensino médio",
    "Estabelecimentos de Saúde SUS"
]

pop_col = "População no último censo"

for coluna in colunas_per_capita:
    df_dados[coluna] = (
        df_dados[coluna].astype(float)
        / df_dados[pop_col].astype(float)
    )

In [31]:
df_dados['Área urbanizada']  = df_dados['Área urbanizada'].astype(float) /df_dados['Área da unidade territorial'].astype(float)

In [32]:
%%capture
for coluna in colunas_numericas:
  df_dados.loc[:, coluna] = categorizar_valores(list(df_dados[coluna]))

In [33]:
colunas_nao_contar = [
    "estado_id",
    "municipio_id",
    "Código da Emenda",
    "Valor Recebido",
    'Ano/Mês',
    # "População no último censo",
     #"População quilombola", #####################################
     #"População indígena", #####################################
    # "Densidade demográfica",
    "Nome masculino mais popular",
    "Nome feminino mais popular",
    "Sobrenome mais popular",
    # "Salário médio mensal dos trabalhadores formais",
    "Pessoal ocupado em postos de trabalho formais", #####################################
    # "Percentual da população com rendimento nominal mensal per capita de até 1/2 salário mínimo",
    # "Taxa de escolarização de 6 a 14 anos de idade",
    # "IDEB – Anos iniciais do ensino fundamental (Rede pública)",
    # "IDEB – Anos finais do ensino fundamental (Rede pública)",
   # "Matrículas no ensino fundamental", #####################################
   "Matrículas no ensino médio", #####################################
    "Docentes no ensino fundamental", #####################################
    "Docentes no ensino médio", #####################################
    #"Número de estabelecimentos de ensino fundamental", #####################################
    "Número de estabelecimentos de ensino médio", #####################################
    # "PIB per capita",
    # "Índice de Desenvolvimento Humano Municipal (IDHM)",
    "Total de receitas brutas realizadas",
    "Transferências correntes (Percentual em relação às receitas correntes brutas realizadas)",
    "Total de despesas brutas empenhadas",
    # "Mortalidade Infantil",
    # "Internações por diarreia pelo SUS",
     #"Estabelecimentos de Saúde SUS", #####################################
    # "Área urbanizada",
    # "Esgotamento sanitário por rede geral, rede pluvial ou fossa ligada à rede",
    # "Arborização de vias públicas",
    # "Urbanização de vias públicas",
    "População exposta ao risco",
    "Bioma predominante",
    "Sistema Costeiro-Marinho",
    "Área da unidade territorial",
    "Hierarquia urbana",
    #  "Região de Influência",
    "Região intermediária",
    # "Região imediata",
    "Mesorregião",
    "Microrregião"
]

In [34]:
df_dados['valor_ementa_alto'] = df_dados['Valor Recebido'].apply(lambda x: x == 'valor_alto')

In [35]:
df_dados['valor_ementa_baixo'] = df_dados['Valor Recebido'].apply(lambda x: x == 'valor_baixo')

# Criando subgrupos sem avaliar o ano

## valor alto

In [50]:
%%capture
!pip install pysubgroup

In [61]:
import pysubgroup as ps

target = ps.BinaryTarget('valor_ementa_alto', True)

searchspace = ps.create_selectors(df_dados, ignore=colunas_nao_contar + ['valor_ementa_alto'] + ['valor_ementa_baixo'])

task = ps.SubgroupDiscoveryTask(
    df_dados,
    target,
    searchspace,
    result_set_size=20,
    depth=3,
    qf=ps.StandardQF(a=0.5),
    min_quality=0.01
)

result = ps.BeamSearch(beam_width=20).execute(task)


In [62]:
result.to_dataframe()[['subgroup', 'quality']]

,subgroup,quality
0,IDEB – Anos finais do ensino fundamental (Rede...,0.070949
1,IDEB – Anos finais do ensino fundamental (Rede...,0.070815
2,PIB per capita=='valor_baixo' AND Percentual d...,0.064598
3,Percentual da população com rendimento nominal...,0.064588
4,Percentual da população com rendimento nominal...,0.064378
5,IDEB – Anos iniciais do ensino fundamental (Re...,0.064264
6,Arborização de vias públicas=='valor_medio' AN...,0.064053
7,"Esgotamento sanitário por rede geral, rede plu...",0.063847
8,Estabelecimentos de Saúde SUS=='valor_baixo' A...,0.063747
9,IDEB – Anos finais do ensino fundamental (Rede...,0.062916


## Valor baixo

In [63]:
import pysubgroup as ps

target_2 = ps.BinaryTarget('valor_ementa_baixo', True)

searchspace_2 = ps.create_selectors(df_dados, ignore=colunas_nao_contar + ['valor_ementa_alto'] + ['valor_ementa_baixo'])

task_2 = ps.SubgroupDiscoveryTask(
    df_dados,
    target_2,
    searchspace_2,
    result_set_size=20,
    depth=3,
    qf=ps.StandardQF(a=0.5),
    min_quality=0.01
)

result_2 = ps.BeamSearch(beam_width=30).execute(task_2)


In [64]:
result_2.to_dataframe()[['subgroup', 'quality']]

,subgroup,quality
0,Densidade demográfica=='valor_alto' AND Área u...,0.096984
1,Densidade demográfica=='valor_alto' AND Popula...,0.096984
2,Área urbanizada=='valor_alto',0.096939
3,População quilombola=='valor_baixo' AND Área u...,0.096939
4,População no último censo=='valor_alto' AND Ár...,0.088173
5,População no último censo=='valor_alto' AND Po...,0.088173
6,Densidade demográfica=='valor_alto' AND Popula...,0.087673
7,Número de estabelecimentos de ensino fundament...,0.084707
8,Número de estabelecimentos de ensino fundament...,0.084707
9,Internações por diarreia pelo SUS=='valor_medi...,0.084347


# Criando subgrupos avaliando a região

In [65]:
colunas_nao_contar_2 = [
    "estado_id",
    "municipio_id",
    "Código da Emenda",
    "Valor Recebido",
    'Ano/Mês',
    "População no último censo",
     "População quilombola", #####################################
     "População indígena", #####################################
    "Densidade demográfica",
    "Nome masculino mais popular",
    "Nome feminino mais popular",
    "Sobrenome mais popular",
     "Salário médio mensal dos trabalhadores formais",
    "Pessoal ocupado em postos de trabalho formais", #####################################
    "Percentual da população com rendimento nominal mensal per capita de até 1/2 salário mínimo",
    "Taxa de escolarização de 6 a 14 anos de idade",
    "IDEB – Anos iniciais do ensino fundamental (Rede pública)",
    "IDEB – Anos finais do ensino fundamental (Rede pública)",
   "Matrículas no ensino fundamental", #####################################
   "Matrículas no ensino médio", #####################################
    "Docentes no ensino fundamental", #####################################
    "Docentes no ensino médio", #####################################
    "Número de estabelecimentos de ensino fundamental", #####################################
    "Número de estabelecimentos de ensino médio", #####################################
     "PIB per capita",
     "Índice de Desenvolvimento Humano Municipal (IDHM)",
    "Total de receitas brutas realizadas",
    "Transferências correntes (Percentual em relação às receitas correntes brutas realizadas)",
    "Total de despesas brutas empenhadas",
    "Mortalidade Infantil",
    "Internações por diarreia pelo SUS",
     "Estabelecimentos de Saúde SUS", #####################################
    "Área urbanizada",
    "Esgotamento sanitário por rede geral, rede pluvial ou fossa ligada à rede",
    "Arborização de vias públicas",
    "Urbanização de vias públicas",
    "População exposta ao risco",
    "Bioma predominante",
    "Sistema Costeiro-Marinho",
    "Área da unidade territorial",
    "Hierarquia urbana",
    #  "Região de Influência",
    "Região intermediária",
    # "Região imediata",
    "Mesorregião",
    "Microrregião"
]

## valor alto

In [66]:
import pysubgroup as ps

target_3 = ps.BinaryTarget('valor_ementa_alto', True)

searchspace_3 = ps.create_selectors(df_dados, ignore=colunas_nao_contar_2 + ['valor_ementa_alto'] + ['valor_ementa_baixo'])

task_3 = ps.SubgroupDiscoveryTask(
    df_dados,
    target_3,
    searchspace_3,
    result_set_size=20,
    depth=3,
    qf=ps.StandardQF(a=0.5),
    min_quality=0.01
)

result_3 = ps.BeamSearch(beam_width=40).execute(task_3)


In [67]:
result_3.to_dataframe()[['subgroup', 'quality']]

,subgroup,quality
0,Região imediata=='Distrito Federal',0.060744
1,Região de Influência=='Arranjo Populacional de...,0.060744
2,Região de Influência=='Arranjo Populacional de...,0.060307
3,Região de Influência=='Arranjo Populacional de...,0.022013
4,Região de Influência=='Boa Vista - Capital Reg...,0.020047
5,Região de Influência=='Arranjo Populacional de...,0.020025
6,Região de Influência=='Manaus - Metrópole (1C)',0.016970
7,Região de Influência=='Arranjo Populacional de...,0.016439
8,Região de Influência=='Arranjo Populacional de...,0.016310
9,Região de Influência=='Rio Branco - Capital Re...,0.014579


## Valor baixo

In [72]:
len(searchspace_4)

1324

In [70]:
import pysubgroup as ps

target_4 = ps.BinaryTarget('valor_ementa_baixo', True)

searchspace_4 = ps.create_selectors(df_dados, ignore=colunas_nao_contar_2 + ['valor_ementa_alto'] + ['valor_ementa_baixo'])

task_4 = ps.SubgroupDiscoveryTask(
    df_dados,
    target_4,
    searchspace_4,
    result_set_size=20,
    depth=3,
    qf=ps.StandardQF(a=0.5),
    min_quality=0.02
)

result_4 = ps.BeamSearch(beam_width=20).execute(task_4)


KeyboardInterrupt: 

In [ ]:
result_4.to_dataframe()[['subgroup', 'quality']]

# Criando subgrupos avaliando região por ano

In [ ]:
def get_ano(ano_mes):
  ano = str(ano_mes)[:4]
  return int(ano)

In [ ]:
df_dados_copy = df_dados.copy()
df_dados_copy['ano'] = df_dados_copy['Ano/Mês'].apply(get_ano)

In [ ]:
df_dados_antes_2021 = df_dados_copy[df_dados_copy['ano'] < 2021]
df_dados_depois_igual_2021 = df_dados_copy[df_dados_copy['ano'] >= 2021]

In [ ]:
len(df_dados_antes_2021),len(df_dados_depois_igual_2021)

## Antes de 2021

### valor alto

In [ ]:
import pysubgroup as ps

target_3 = ps.BinaryTarget(
    'valor_ementa_alto',
    True
)

In [ ]:
searchspace_3 = ps.create_selectors(
    df_dados_antes_2021,
    ignore=colunas_nao_contar_2 + ['valor_ementa_alto'] + ['valor_ementa_baixo'] +  ['ano']
)

In [ ]:
task_3 = ps.SubgroupDiscoveryTask(
    df_dados_antes_2021,
    target_3,
    searchspace_3,
    result_set_size=20,
    depth=4,
    qf=ps.StandardQF(1),
     min_quality=0.01
)

In [ ]:
len(searchspace_3)

In [ ]:
result_3 = ps.DFS().execute(task_3)


In [ ]:
result_3.to_dataframe()[['subgroup', 'quality']]

### valor baixo

In [ ]:
import pysubgroup as ps

target_4 = ps.BinaryTarget(
    'valor_ementa_baixo',
    True
)

In [ ]:
searchspace_4 = ps.create_selectors(
    df_dados_antes_2021,
    ignore=colunas_nao_contar_2 + ['valor_ementa_alto'] + ['valor_ementa_baixo'] +  ['ano']
)

In [ ]:
task_4 = ps.SubgroupDiscoveryTask(
    df_dados_antes_2021,
    target_4,
    searchspace_4,
    result_set_size=20,
    depth=4,
    qf=ps.StandardQF(1),
     min_quality=0.01
)

In [ ]:
len(searchspace_4)

In [ ]:
result_4 = ps.DFS().execute(task_4)


In [ ]:
result_4.to_dataframe()[['subgroup', 'quality']]

## 2021 para frente

### valor alto

In [ ]:
import pysubgroup as ps

target_5 = ps.BinaryTarget(
    'valor_ementa_alto',
    True
)

In [ ]:
searchspace_5 = ps.create_selectors(
    df_dados_depois_igual_2021,
    ignore=colunas_nao_contar_2 + ['valor_ementa_alto'] + ['valor_ementa_baixo'] +  ['ano']
)

In [ ]:
task_5 = ps.SubgroupDiscoveryTask(
    df_dados_depois_igual_2021,
    target_5,
    searchspace_5,
    result_set_size=20,
    depth=4,
    qf=ps.StandardQF(1),
     min_quality=0.01
)

In [ ]:
len(searchspace_5)

In [ ]:
result_5 = ps.DFS().execute(task_5)

In [ ]:
result_5.to_dataframe()[['subgroup', 'quality']]

### valor baixo

In [ ]:
import pysubgroup as ps

target_6 = ps.BinaryTarget(
    'valor_ementa_baixo',
    True
)

In [ ]:
searchspace_6 = ps.create_selectors(
    df_dados_depois_igual_2021,
    ignore=colunas_nao_contar_2 + ['valor_ementa_alto'] + ['valor_ementa_baixo'] +  ['ano']
)

In [ ]:
task_6 = ps.SubgroupDiscoveryTask(
    df_dados_depois_igual_2021,
    target_6,
    searchspace_6,
    result_set_size=20,
    depth=4,
    qf=ps.StandardQF(1),
     min_quality=0.01
)

In [ ]:
len(searchspace_6)

In [ ]:
result_6 = ps.DFS().execute(task_6)


In [ ]:
result_6.to_dataframe()[['subgroup', 'quality']]